# 1. Import Libraries


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import numpy as np
from datetime import datetime

# 2. Constants


In [ ]:
url = 'https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29'
table_attribs = ['Country', 'GDP_USD_millions']
db_name = 'World_Economies.db'
table_name = 'Countries_by_GDP'
csv_path = '/home/project/Countries_by_GDP.csv'
conn = sqlite3.connect(db_name)
log_file = "./etl_project_log.txt"

# 3. Extract Data


In [ ]:
# Loading the entire webpage for Webscraping
html_page = requests.get(url).text

In [ ]:
html_page

In [ ]:
# Parse the text in the HTML format using BeautifulSoup
data = BeautifulSoup(html_page, 'html.parser')

In [ ]:
data

In [ ]:
# Loop to extract data in table (tbody)
tables = data.find_all('tbody')

In [ ]:
tables

In [ ]:
rows = tables[2].find_all('tr')

In [ ]:
rows

In [ ]:
df = pd.DataFrame(columns=table_attribs)
for row in rows:
        col = row.find_all('td')
        if len(col)!=0:
            if col[0].find('a') is not None and '—' not in col[2]:
                data_dict = {table_attribs[0]: col[0].a.contents[0],
                             table_attribs[1]: col[2].contents[0]}
                df1 = pd.DataFrame(data_dict, index=[0])
                df = pd.concat([df,df1], ignore_index=True)

In [ ]:
df

### Final extracting data function


In [ ]:
def extract(url, table_attribs):
	''' This function extracts the required
	information from the website and saves it to a dataframe. The
	function returns the dataframe for further processing. '''

	# Loading the entire webpage for Webscraping
	html_page = requests.get(url).text

	# Parse the text in the HTML format using BeautifulSoup
	data = BeautifulSoup(html_page, 'html.parser')

	# Loop to extract data in table (tbody)
	tables = data.find_all('tbody')

	# Select 3rd table
	rows = tables[2].find_all('tr')

	# Create dataframe with table_attribs columns
	df = pd.DataFrame(columns=table_attribs)

	for row in rows:
			col = row.find_all('td')
			if len(col)!=0:
				if col[0].find('a') is not None and '—' not in col[2]:
					data_dict = {table_attribs[0]: col[0].a.contents[0],
								table_attribs[1]: col[2].contents[0]}
					df1 = pd.DataFrame(data_dict, index=[0])
					df = pd.concat([df,df1], ignore_index=True)
		
	return df

In [ ]:
df = extract(url, table_attribs)

In [ ]:
df

# 4. Transform Data


In [ ]:
df.info()

In [ ]:
transformed_df = df.copy(deep=True)

In [ ]:
# Remove ","
transformed_df['GDP_USD_millions'] = transformed_df['GDP_USD_millions'].astype(str).str.replace(",", "")
transformed_df

In [ ]:
# Change the data type of GDP_USD_millions from object to numeric
# 1. Using astype(float) This is the simplest method when you are sure all values are numeric strings.
# 2. Using pd.to_numeric() This method is safer for messy data. With errors='coerce', non-convertible values become NaN.
transformed_df['GDP_USD_millions'] = pd.to_numeric(transformed_df['GDP_USD_millions'], errors="coerce")
print(f"Data type after changed format in GDP_USD_millions: {transformed_df['GDP_USD_millions'].dtypes}")

In [ ]:
# Change GDP_USD_millions to GDP_USD_billions by dividing ll these values by 1000 and round it to 2 decimal places.
transformed_df['GDP_USD_millions'] = (transformed_df['GDP_USD_millions']/1000).round(2)
transformed_df

In [ ]:
# Rename
transformed_df.rename(columns={"GDP_USD_millions":"GDP_USD_billions"}, inplace=True)
transformed_df

In [ ]:
type(transformed_df)

In [ ]:
def transform(df):
	''' This function converts the GDP information from Currency
	format to float value, transforms the information of GDP from
	USD (Millions) to USD (Billions) rounding to 2 decimal places.
	The function returns the transformed dataframe.'''

	# Remove ","
	df['GDP_USD_millions'] = df['GDP_USD_millions'].astype(str).str.replace(",", "")

	# Change the data type of GDP_USD_millions from object to numeric
	df['GDP_USD_millions'] = pd.to_numeric(df['GDP_USD_millions'], errors="coerce")

	# Change GDP_USD_millions to GDP_USD_billions by dividing ll these values by 1000 and round it to 2 decimal places.
	df['GDP_USD_millions'] = (df['GDP_USD_millions']/1000).round(2)

	# Rename
	df.rename(columns={"GDP_USD_millions":"GDP_USD_billions"}, inplace=True)

	return df

In [ ]:
test_transform = transform(df)

In [ ]:
test_transform

# 5. Loading Data

In [ ]:
# Save to CSV files
transformed_df.to_csv("Countries_by_GDP.csv", index=False)

In [ ]:
def load_to_csv(df, csv_path):
	''' This function saves the final dataframe as a `CSV` file 
	in the provided path. Function returns nothing.'''
	df.to_csv(csv_path, index=False)

In [ ]:
# Save to db
conn = sqlite3.connect(db_name)
df.to_sql(table_name, conn, if_exists='replace', index=False)
conn.close()

In [ ]:
def load_to_db(df, sql_connection, table_name):
	''' This function saves the final dataframe as a database table
	with the provided name. Function returns nothing.'''
	df.to_sql(table_name, sql_connection, if_exists='replace', index=False)

In [ ]:
test_load_db = load_to_db(test_transform, conn, table_name)

# 6. Querying the database table

In [ ]:
def run_query(query_statement, sql_connection):
	''' This function runs the stated query on the database table and
	prints the output on the terminal. Function returns nothing. '''
	df = pd.read_sql(query_statement, sql_connection)
	return df

In [ ]:
query = f"""SELECT * FROM {table_name} WHERE GDP_USD_billions >= 100"""
test_query = run_query(query, conn)
test_query

# 7. Logging progress

In [ ]:
def log_progress(message):
	''' This function logs the mentioned message at a given stage of the code execution to a log file. Function returns nothing'''
	timestamp_format = '%Y-%h-%d-%H:%M:%S'
	now = datetime.now()
	timestamp = now.strftime(timestamp_format)
	with open(log_file, "a") as f:
		f.write(timestamp + ' : ' + message + '\n')

In [ ]:
log_progress("test")